# 02. Pollution & DSC Scoring (Image Cell)

**Phase 1**: 5종 polluter × 6 level × 3 데이터셋 → DSC 점수 측정.

split-first 원칙: train/test 분할 → train에만 polluter 적용. (torchvision의 train/test split 그대로 사용 — 별도 split 불필요)

---

In [ ]:
# ============================================================
# 0-1. 환경 + 데이터 로드
# ============================================================
from google.colab import drive; drive.mount('/content/drive')
import os, sys, json
import numpy as np
import pandas as pd
import torch
import torchvision

BASE = '/content/drive/MyDrive/capstone/dsc'
RESULTS_DIR = f'{BASE}/results'
DATA_DIR = f'{BASE}/data/image'
POLLUTED_DIR = f'{BASE}/data/image_polluted'
os.makedirs(POLLUTED_DIR, exist_ok=True)

if BASE not in sys.path:
    sys.path.insert(0, BASE)

%pip install -q timm imagehash opencv-python-headless

In [ ]:
# ============================================================
# 0-2. 사전등록 (DATASETS, POLLUTION_LEVELS)
# ============================================================
DATASETS = {
    'CIFAR10': {'loader': 'CIFAR10', 'n_classes': 10, 'image_size': 32, 'channels': 3},
    'FashionMNIST': {'loader': 'FashionMNIST', 'n_classes': 10, 'image_size': 28, 'channels': 1},
    'Flowers102': {'loader': 'Flowers102', 'n_classes': 102, 'image_size': 224, 'channels': 3},
}
POLLUTION_LEVELS = [0.1, 0.25, 0.5, 0.75, 0.9, 0.95]
RANDOM_SEED = 42
SAMPLE_CAP = 5000  # 폴루션 후 DSC 계산 시 sample_cap

def load_train(ds_name):
    if ds_name == 'CIFAR10':
        return torchvision.datasets.CIFAR10(f'{DATA_DIR}/CIFAR10', train=True, download=True)
    if ds_name == 'FashionMNIST':
        return torchvision.datasets.FashionMNIST(f'{DATA_DIR}/FashionMNIST', train=True, download=True)
    if ds_name == 'Flowers102':
        return torchvision.datasets.Flowers102(f'{DATA_DIR}/Flowers102', split='train', download=True)

print(f'데이터셋: {list(DATASETS.keys())}, 강도: {POLLUTION_LEVELS}')

## 1. Polluter 5종 + DSC 점수

In [ ]:
# ============================================================
# 1-1. polluter import + DSC import
# ============================================================
from dsc_framework import compute_dsc_image
from dsc_framework.image_polluters import (
    CompletenessImagePolluter, NoiseInjectionPolluter, BlurPolluter,
    ClassBalanceImagePolluter, LabelSwapPolluter,
)

def create_polluters(level, seed=RANDOM_SEED):
    return [
        ('completeness_image', CompletenessImagePolluter(level=level, random_seed=seed)),
        ('noise_injection', NoiseInjectionPolluter(level=level, random_seed=seed)),
        ('blur', BlurPolluter(level=level, random_seed=seed)),
        ('class_balance', ClassBalanceImagePolluter(level=level, random_seed=seed)),
        ('label_swap', LabelSwapPolluter(level=level, random_seed=seed)),
    ]
print('Polluter 5종 정의 완료')

In [ ]:
# ============================================================
# 1-2. dataset → numpy 변환 + 폴루션 적용 + DSC
# ============================================================
def dataset_to_arrays(ds, sample_cap=None, random_state=1):
    images, labels = [], []
    n = len(ds) if sample_cap is None else min(len(ds), sample_cap)
    rng = np.random.RandomState(random_state)
    idx = rng.permutation(len(ds))[:n] if sample_cap else range(n)
    for i in idx:
        img, lbl = ds[i]
        images.append(np.array(img))
        labels.append(int(lbl))
    return images, labels


from time import time

dsc_rows = []
total_start = time()

for ds_name in DATASETS:
    print(f'\n=== {ds_name} ===')
    train_ds = load_train(ds_name)
    images_clean, labels_clean = dataset_to_arrays(train_ds, sample_cap=SAMPLE_CAP, random_state=1)
    print(f'  loaded {len(images_clean)} images')

    # baseline DSC
    res_base = compute_dsc_image(images_clean, labels_clean, sample_cap=SAMPLE_CAP)
    print(f'  baseline DSC = {res_base["score"]} ({res_base["grade"]})')
    dsc_rows.append({'dataset': ds_name, 'polluter': 'none', 'level': 0.0, **res_base})

    # 폴루션
    for level in POLLUTION_LEVELS:
        for polluter_name, polluter in create_polluters(level):
            t0 = time()
            try:
                pi, pl = polluter.pollute(images_clean, labels_clean)
                res_p = compute_dsc_image(pi, pl, sample_cap=SAMPLE_CAP)
                # 폴루션 데이터를 디스크에 저장 (03 노트북이 다시 사용)
                pol_dir = f'{POLLUTED_DIR}/{ds_name}/{polluter_name}_{int(level*100)}'
                os.makedirs(pol_dir, exist_ok=True)
                np.savez_compressed(f'{pol_dir}/data.npz',
                                    images=np.array([np.asarray(img) for img in pi], dtype=object),
                                    labels=np.array(pl))
                dsc_rows.append({'dataset': ds_name, 'polluter': polluter_name, 'level': level, **res_p})
                elapsed = time() - t0
                print(f'  {polluter_name:<22s} L={level:.2f}  DSC={res_p["score"]:6.2f}  Δ={res_p["score"]-res_base["score"]:+.2f}  ({elapsed:.0f}s)')
            except Exception as e:
                print(f'  {polluter_name:<22s} L={level:.2f}  ERROR: {e}')

print(f'\n총 {len(dsc_rows)}건 ({time() - total_start:.0f}초)')

In [ ]:
# ============================================================
# 2-3. 결과 저장
# ============================================================
df_dsc = pd.DataFrame(dsc_rows)
out_path = f'{RESULTS_DIR}/dsc_scores_image.csv'
df_dsc.to_csv(out_path, index=False)
print(f'DSC 점수 저장: {out_path} (총 {len(df_dsc)}건)')
print('--- 노트북 02 이미지 cell 완료 ---')
df_dsc.head(15)